# Optional lab - Do word objectives shape different profiles?

This is the original larger comparison, separated from
[the small embedding lesson](03A-training-skeleton-and-word-objectives.ipynb#encoder-decoder-preview).
It is **not** a continuation of that model or its weights.

Run All from a fresh chapter kernel; no CBOW, Skip-gram or reader/writer run
from another notebook is required. Concrete model implementations are reused
from the tested Python module; their learning definitions remain visible in the main lesson.

| Scope | Evidence |
|---|---|
| Built and measured | Three objectives, identical starting rows, story-level split, 16 epochs, fixed-camera plots, full-row similarities, neighbours and best/final checks |
| Explained | Geometry versus prediction quality, overfitting, ambiguous CBOW answers |
| Out of scope | A fair architecture ranking, unseen general-language quality, attention |

Keep the awkward later epochs: useful geometry and held-out predictions need
not improve together. All three original training experiments remain here.


In [ ]:
# %% Setup - Run the comparison independently of the main notebook
from pathlib import Path
import sys
import time
import matplotlib.pyplot as plt
import torch

CHAPTER_DIRECTORY = next(
    candidate for candidate in (
        Path.cwd(), Path.cwd() / "learning" / "03-language-model-foundations" / "03-language-models",
    ) if (candidate / "real_text_embeddings.py").is_file()
)
if str(CHAPTER_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(CHAPTER_DIRECTORY))
from language_model import CBOW, SkipGram, EncoderDecoderLM
from encoder_decoder_feedback import responsive_animation_html
torch.set_num_threads(1)
notebook_started = time.perf_counter()


## 1 - Load the independent comparison split

<a id="independent-comparison"></a>

Start fresh CBOW, Skip-gram and encoder-decoder models on **8,000 stories**.
Split **80/20 by whole story before vocabulary construction or windows**.
Keep the original settings: 48-number rows with a common start, 40,000 training
windows, 4,000 held-out windows, 16 epochs and seed 71.

The main lesson's 200 reserved stories evaluate only its small model; they are
not an untouched test after these different runs. This writer predicts two
continuation words without the main preview's easy EOS target.

**Predict:** does falling practice loss guarantee better held-out predictions
or useful geometry among the named probe words?


In [ ]:
# %% 1 - Load the independent comparison split
from pathlib import Path

import real_text_embeddings
from real_text_embeddings import (
    animate_word_maps,
    blank_shortlist,
    load_real_text,
    plot_losses_and_geometry,
    plot_similarity_heatmaps,
    plot_word_maps,
    project_snapshots,
    report_neighbours,
    shortlist_summary,
    train_comparison,
)
from real_text_embeddings import best_versus_final

real_corpus = load_real_text()


## 2 - Same starting rows, different jobs

| Model | Input from each window | Prediction |
|---|---|---|
| CBOW | Four neighbours | Center |
| Skip-gram | Center | One neighbour per pair |
| Reader/writer | First three tokens | Last two tokens in order |

Each gets its own aligned pairs and loader, but the same inherited optimizer
routine. This comparison omits the writer's easy stop target. Epoch zero is the
actual shared initialization.

Target counts and parameter counts differ, so this is not a fair-speed contest
or a causal ranking of architectures. Monitor each objective's held-out curve;
do not rank unlike tasks by raw loss alone.


In [ ]:
# %% 2 - Record all three training histories
REAL_TEXT_EPOCHS = 16
real_results = train_comparison(
    real_corpus, CBOW, SkipGram, EncoderDecoderLM, epochs=REAL_TEXT_EPOCHS,
)


## 3 - Compare practice and held-out trajectories

Inspect the actual curves for divergence. The geometry panel compares within-
group and across-group cosine; group labels are diagnostic only and never enter
training. The best held-out checkpoint need not be the final epoch.


In [ ]:
# %% 3 - Plot recorded loss and geometry
chart_directory = Path(real_text_embeddings.__file__).resolve().parent / "images"
loss_figure = plot_losses_and_geometry(real_results)
loss_figure.savefig(chart_directory / "03a-real-text-loss-and-similarity.png", dpi=130)
plt.show()
plt.close(loss_figure)


![Measured training and held-out loss for all three objectives, with within-group and across-group cosine similarity below.](images/03a-real-text-loss-and-similarity.png)
*Saved from the measured run; running the cell above regenerates this chart.*

Here's the complaint the tiny corpus hid: **we can get better at the practice questions and
worse at new passages at the same time.** The best held-out epochs were 5 for CBOW, 3 for
Skip-gram, and 4 for the writer. We deliberately show all 16 epochs instead of cropping off
the awkward part. These held-out stories are monitoring data, not an untouched final test set.

Now look at the writer's bottom panel. Its gap shoots up fastest (about **+0.54** by
epoch 2, higher than either other model ever gets), then slides down for the rest of training.
Hang on to that; we'll come back to it.


## 4 - Keep the projection camera fixed

One PCA projection and fixed axes cover all epochs and objectives. It is a view
of a 48-dimensional trajectory, not the whole representation.


In [ ]:
# %% 4 - Project all histories through one camera
real_probe_words, real_coordinates, real_pca_fraction = project_snapshots(
    real_corpus, real_results,
)
map_figure = plot_word_maps(real_probe_words, real_coordinates, real_results)
map_figure.savefig(chart_directory / "03a-real-text-word-maps.png", dpi=130)
plt.show()
plt.close(map_figure)


![The same sixteen words at initialization and after training for CBOW, Skip-gram, and the encoder-decoder, using one frozen PCA projection.](images/03a-real-text-word-maps.png)
*The colors name our inspection groups. Training never receives those colors.*


## 5 - Check full-row similarity too

These heatmaps compare all 48 coordinates with a shared scale. The omitted
diagonal is self-similarity. Inspect both within-group blocks and cross-group
values rather than accepting tidy islands in a two-dimensional map.


In [ ]:
# %% 5 - Compare complete stored rows
similarity_figure = plot_similarity_heatmaps(real_corpus, real_results)
similarity_figure.savefig(chart_directory / "03a-real-text-cosine-heatmaps.png", dpi=130)
plt.show()
plt.close(similarity_figure)


![Cosine similarity heatmaps for the common random start and each trained model, computed in all 48 embedding dimensions with a shared scale.](images/03a-real-text-cosine-heatmaps.png)
*This checks the rows directly. It doesn't depend on the PCA camera.*


## 6 - Optional recorded playback

Each frame uses a saved epoch, not invented interpolation. The static plots
above remain the evidence when inline playback is unavailable.


In [ ]:
# %% 6 - Play the recorded snapshots
real_animation = animate_word_maps(real_probe_words, real_coordinates, real_results)
real_animation_html = responsive_animation_html(real_animation)
if "get_ipython" in globals():
    from IPython.display import HTML, display

    display(HTML(real_animation_html))
else:
    print(f"Recorded animation rendered ({len(real_animation_html):,} HTML characters).")


## 7 - Inspect the neighbours behind the picture

Report nearest probe neighbours and full-vocabulary neighbours before and after
training. A small named probe set is a sanity check, not overall semantic quality.
Compare actual pairs against across-group similarity to detect a general collapse
of all rows into one direction.


In [ ]:
# %% 7 - Print actual neighbour changes
report_neighbours(real_corpus, real_results)


**Your turn:** change only `probe_pair`, without retraining. Predict the direction
of its change and compare the result with the across-group baseline.


In [ ]:
# %% 7 - Your turn: Inspect another pair
probe_pair = ("mom", "dad")  # CHANGE THIS: try ("mom", "green"), without retraining
for objective, record in real_results.items():
    pair_ids = [real_corpus["word_ids"][word] for word in probe_pair]
    pair_scores = []
    for rows in (record["snapshots"][0], record["snapshots"][-1]):
        pair_vectors = torch.nn.functional.normalize(rows[pair_ids], dim=1)
        pair_scores.append((pair_vectors[0] @ pair_vectors[1]).item())
    print(f"{objective}: {' / '.join(probe_pair)} cosine "
          f"{pair_scores[0]:+.3f} -> {pair_scores[1]:+.3f}; "
          f"final across-group mean {record['across'][-1]:+.3f}")


## 8 - A missing word can have several plausible answers

Use CBOW's best held-out checkpoint. Inspect the full shortlist, correct-answer
frequency, confidence, and row similarity rather than showing only a friendly
top guess. Similar usage does not imply synonyms or interchangeability.


In [ ]:
# %% 8 - Check held-out answer shortlists
cbow_best = real_results["CBOW"]["best_model"]
for left, right in [("was so", "to have"), ("a little", "named tom"),
                    ("played with", "toys and"), ("they went", "the park")]:
    menu = blank_shortlist(cbow_best, real_corpus, left, right)
    print(f"{left} ___ {right} -> " + ", ".join(f"{word} {chance:.0%}" for word, chance in menu))

shortlist = shortlist_summary(cbow_best, real_corpus)
print(f"\nAll {shortlist['blanks']:,} held-out blanks with a known answer:")
print(f"  right answer is the number-one guess:   {shortlist['first_right']:.1%}")
print(f"  right answer is somewhere in the top 5: {shortlist['shortlist_right']:.1%}")
print(f"  average share given to the first guess: {shortlist['first_share']:.1%}")
print(f"  how alike the 5 shortlisted words' rows are: {shortlist['shortlist_similarity']:+.3f}")
print(f"  how alike 5 random words' rows are:          {shortlist['random_similarity']:+.3f}")



## 9 - Compare the writer's best checkpoint with its last

Does a weaker final geometry plot mean the writer never learned useful rows?
Read its recorded best-versus-final values. A fading gap after held-out loss
stops improving is evidence about this run, not proof that extra recurrent
parameters caused it; objectives, target counts, and computation also differ.


In [ ]:
# %% 9 - Check best versus final without retraining
writer_check = best_versus_final(real_results)
writer_record = writer_check["Encoder-decoder"]
if writer_record["final_gap"] < writer_record["best_gap"]:
    print("The writer's group separation faded after its best held-out checkpoint.")
else:
    print("This run did not show fading group separation; judge the recorded losses too.")


The message branches on the recorded outcome. Extra-seed runs below are off by
default and cost another full comparison each; they provide no evidence unless run.


In [ ]:
# %% 9 - Optional extra-seed robustness check
RUN_SEED_CHECK = False
if RUN_SEED_CHECK:
    for check_seed in (5, 123):
        print(f"--- seed {check_seed}")
        best_versus_final(train_comparison(
            real_corpus, CBOW, SkipGram, EncoderDecoderLM,
            epochs=REAL_TEXT_EPOCHS, seed=check_seed, verbose=False,
        ))
else:
    print("Skipped. Set RUN_SEED_CHECK = True to retrain with seeds 5 and 123.")
print(f"Whole optional lab: {time.perf_counter() - notebook_started:.1f}s "
      "(cache verified, one CPU thread; first download is extra).")


## What this comparison establishes

Each objective learns its own input profiles from prediction, starting with
the same rows. Read within-objective held-out curves alongside geometry.
PCA pictures need full-row checks; nearest neighbours need unrelated controls.
Parameter budgets and answer counts differ, so lower raw loss is not an
architecture ranking. Seed checks remain optional and provide no evidence
unless actually run.


## Optional reference - Correct previous words during practice

<a id="teacher-forcing-reference"></a>

The reader has read **the sleepy cat**. You want the writer to learn
**sat -> on -> stop**. It will make mistakes at first. If it guesses `and`
instead of `sat`, should its next practice question start with `and`, or
should you still let it practise what comes after the real word `sat`?

For this training run, choose the second option: supply the **correct previous word**.
That input rule is called **teacher forcing**. It is important in next-word
training, not a special GRU learning algorithm. Backpropagation and the optimizer
still do the learning.

#### The teacher supplies context, not the current answer

The writer still predicts **one word** at each step:

| Step | Word shown now | Writer must predict | Where does its hidden state come from? |
|---|---|---|---|
| 1 | `<bos>`: start writing | `sat` | The reader's final state |
| 2 | Correct previous `sat` | `on` | Writer state after step 1 |
| 3 | Correct previous `on` | `<eos>`: stop | Writer state after step 2 |

`sat` is **not the word currently being guessed** at step 2; `on` is.
If step 1 guessed `and`, that guess stays wrong and contributes to the loss.
Showing `sat` afterward does not fix that guess or reset the hidden state.
It gives the next step a clean question: "after the source and `sat`, what comes next?"

![Same fitted cat model with correct previous-word inputs during practice versus its own guesses during generation; each guess is still checked and hidden state carries in both rows.](images/03a-correct-previous-word-vs-own-guess.png)

<details>
<summary>Watch which word becomes the next input</summary>

![Saved comparison of correct-previous-word practice and own-guess generation, highlighting each actual decoder step without changing the model weights.](images/03a-correct-previous-word-vs-own-guess.gif)

[Pause or step through the comparison](images/03a-correct-previous-word-vs-own-guess.html).

</details>

**Is that cheating?** Only if we supplied `sat` to the step being asked to guess
`sat`. We do not. The model reads left to right, so a later input cannot help
an earlier guess. The two slices in the reader/writer's forward method line up like this:

```text
Stored target:   <bos>  sat  on  <eos>
Shown as input:  <bos>  sat  on          (drop the last item)
Checked answer: sat    on   <eos>       (drop the start item)
```

#### What actually changes during practice?

For each batch of examples:

1. Read each source and carry its final state into the writer.
2. Read the correct previous words and produce a score for every possible next word.
3. Check all three next-word answers. **Cross-entropy loss** is the penalty for
   assigning too little probability to the correct answers; it checks the scores,
   not just the highest guess.
4. Backward feedback calculates which weights contributed to those penalties.
   One optimizer step then updates the shared embedding table, both GRUs and
   the word-prediction head.

Repeat for **six epochs**: six complete passes through the practice examples,
not just six weight updates. Hidden state changes as words arrive; weights
change after the batch is checked.

This reference describes the main lesson's six-epoch cat preview, including
its stop target; the independent comparison above omits that easy target.


### Reading the small preview's diagnostics

In the main lesson's small preview, all next-word checks use **correct previous words**. Read the
training and validation results as different questions:

| Readout | Plain-English question |
|---|---|
| Practice loss | On examples used for learning, did the model put more probability on the right next words? Lower is better. |
| `P(sat \| source, <bos>)` | For our familiar cat question, how much probability goes to `sat`? An increase need not make `sat` the highest guess; read the actual before/after numbers. |
| Training accuracy | With correct earlier words supplied, how often was the highest next-word guess right on the first 4,096 practice windows? |
| Accuracy without stop | How often were the two content words right, leaving out the relatively easy stop answer? |
| Reserved-story validation | Did next-word prediction improve on stories not used for weight updates? The before/after comparison uses those same reserved questions. |
| Embedding row change | Did a stored word profile change from its starting values? Movement alone does not prove that a feature acquired a useful meaning. |

The saved cat probe is still wrong after six epochs; a lower overall loss does
not guarantee success on every familiar question. The second printed probe
uses source `cat sat on` and correct previous `the` to ask for `mat`.

Every example in this preview has stop as its third answer. Getting only stop
right already gives one out of three checks right. That is why the content-only
accuracy is reported separately.

The reserved-story comparison is a real held-out check, but still **not
free-running generation**: correct earlier words are supplied there too.
It does not measure whether the model can write an entire continuation by itself.
Mistakes can become new inputs when it writes alone; its final readout checks that separately.

The row table shows source words, writer-input words and the start marker.
`cat` can learn through the reader; `<bos>` learns as a writer input.
`<eos>` is different: here it is only a checked answer, never an input lookup.
Its separate output-score weights learn to predict stop while its input
embedding row stays unchanged. That is expected, not a failed update.

<details>
<summary>What was the extra connection check before training?</summary>

Before fitting, a helper briefly uses **copies** to check how learning reaches
the source words. Cutting only the backward connection from writer to reader
leaves the forward predictions unchanged, but removes feedback to the `cat` input row,
which only the reader looks up in this question. This is a wiring check, not another fit or a change to the real model.
The recorder likewise only saves and reports snapshots.

</details>
